## Split Overmerged Authors — Phase 1: Zero-Overlap Last Names

Identifies work_authors where the parsed last name of `raw_author_name` has ZERO overlap with the author profile's parsed last name. These are clearly wrong matches (different people) that need to be split by nulling `author_id`.

**Safety filters:**
- Excludes East Asian first/last swaps (Phase 2)
- Excludes work_authors where the work's raw_orcid (from crossref/datacite) matches the author profile's ORCID
- Requires both last names > 2 chars (excludes single-char or empty)
- No substring overlap in either direction

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_phase1 AS
WITH work_parsed AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
    an_w.parsed_name.last AS work_last,
    an_w.parsed_name.first AS work_first,
    an_w.parsed_name.middle AS work_middle
  FROM openalex.works.work_authors wa
  JOIN openalex.authors.author_names an_w ON wa.raw_author_name = an_w.raw_author_name
  WHERE wa.author_id IS NOT NULL
    AND an_w.parsed_name.last IS NOT NULL
    AND LENGTH(an_w.parsed_name.last) > 1
),
-- Per-row work raw_orcid from source metadata (used only for the exclusion below)
work_orcids AS (
  SELECT wb.id AS work_id, pos AS author_sequence, authorship.raw_orcid
  FROM openalex.works.openalex_works_base wb
  LATERAL VIEW POSEXPLODE(wb.authorships) t AS pos, authorship
  WHERE authorship.raw_orcid IS NOT NULL AND authorship.raw_orcid != ''
),
-- Resolved institution IDs per (work_id, author_sequence). Source: work_author_affiliations
-- (BIGINT institution_id). Format aligned to authors_for_matching's URL form so they overlap.
work_inst_ids AS (
  SELECT work_id, author_sequence,
    COLLECT_SET(CONCAT('https://openalex.org/I', CAST(institution_id AS STRING))) AS work_institutions
  FROM openalex.works.work_author_affiliations
  WHERE institution_id IS NOT NULL
  GROUP BY work_id, author_sequence
)
SELECT wp.work_id, wp.author_sequence, wp.author_id, wp.raw_author_name,
  wp.work_last, wp.work_first, wp.work_middle,
  ap.last AS author_last, ap.first AS author_first, ap.middle AS author_middle,
  ap.orcid, ap.works_count,
  COALESCE(wi.work_institutions, ARRAY()) AS work_institutions,
  COALESCE(ap.institution_ids, ARRAY()) AS author_institution_ids
FROM work_parsed wp
JOIN openalex.authors.authors_for_matching ap ON wp.author_id = ap.author_id
LEFT JOIN work_inst_ids wi ON wp.work_id = wi.work_id AND wp.author_sequence = wi.author_sequence
WHERE wp.work_last != ap.last
  AND LENGTH(wp.work_last) > 2 AND LENGTH(ap.last) > 2
  -- No substring overlap in either direction (raw or hyphen-stripped)
  AND INSTR(wp.work_last, ap.last) = 0
  AND INSTR(ap.last, wp.work_last) = 0
  AND INSTR(TRANSLATE(wp.work_last, '-', ''), TRANSLATE(ap.last, '-', '')) = 0
  AND INSTR(TRANSLATE(ap.last, '-', ''), TRANSLATE(wp.work_last, '-', '')) = 0
  -- Exclude East Asian first/last swaps (Phase 2)
  AND NOT (wp.work_last = ap.first AND wp.work_first = ap.last)
  -- Exclude where the work's raw_orcid confirms this author assignment
  AND NOT EXISTS (
    SELECT 1 FROM work_orcids wo
    WHERE wo.work_id = wp.work_id
      AND wo.author_sequence = wp.author_sequence
      AND wo.raw_orcid = ap.orcid
  )

### Cell 2: Validation statistics

In [ ]:
SELECT COUNT(*) AS total_candidates,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id) AS distinct_works,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates_phase1

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT work_id, author_id, raw_author_name,
  work_last, work_first,
  author_last, author_first,
  works_count AS author_works_count
FROM openalex.authors.overmerge_split_candidates_phase1
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_phase1 AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, work_last, author_last,
  current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_phase1

### Cell 5: Execute the split

**WARNING**: This nulls out author_ids. Verify cells 2-3 before running.

**NOTE**: MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically. A separate re-matching run or cutoff relaxation is needed.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_phase1
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue split works for reprocessing by UpdateWorkAuthorships

Push affected work_ids into `curated_work_ids_pending_sync` so the next `UpdateWorkAuthorships` run picks them up and propagates the nulled author_ids through to `work_authorships` → `openalex_works`.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_phase1

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase1 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking (run after MatchAuthors re-processes)

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase1 log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1

### Cell 9: Rollback first-name-match false positives

Phase 1 incorrectly split works for authors who changed their last name (e.g., marriage). When `work_first = author_first` AND at least one corroborating signal matches, the different last name likely reflects a name change, not an overmerge. Restore these records.

**Co-signal requirement:** restore only when first names match AND at least one of:
- work shares any institution with the author profile's affiliations (`work_institutions ∩ author_institution_ids`)
- middle name first letter matches (handles `William` ↔ `W.` initial-vs-full)

**Why the co-signal:** earlier versions restored on first-name match alone (~904K records, ~232K authors). That correctly saved name-change cases like Kyle Demes (formerly Kyle Glenn) but also re-merged unrelated profiles sharing a common first name (e.g., two unrelated "María"s with disjoint last names). Requiring at least one corroborating signal keeps legitimate name-changers protected without re-attracting unrelated profiles.

**Confirmed example:** Kyle Demes (formerly Kyle Glenn), author `5086928770` — same first name + middle initial match (`william` vs `w.`) → restored.

**Note:** ORCID match isn't checked here because Cell 1 already excludes any candidate where the work's `raw_orcid` matches the author profile's ORCID — by construction it can never fire as a positive signal at this stage.

Only updates rows still NULL (won't overwrite re-matches).

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT work_id, author_sequence, FIRST(author_id) AS previous_author_id
  FROM openalex.authors.overmerge_split_candidates_phase1
  WHERE work_first = author_first
    AND work_first IS NOT NULL AND work_first != ''
    AND LENGTH(work_first) > 2
    AND ( ARRAYS_OVERLAP(work_institutions, author_institution_ids)
       OR (work_middle IS NOT NULL AND author_middle IS NOT NULL
           AND LENGTH(TRANSLATE(work_middle, '.', '')) >= 1
           AND LENGTH(TRANSLATE(author_middle, '.', '')) >= 1
           AND SUBSTRING(TRANSLATE(work_middle, '.', ''), 1, 1)
             = SUBSTRING(TRANSLATE(author_middle, '.', ''), 1, 1)) )
  GROUP BY work_id, author_sequence
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id IS NULL THEN
  UPDATE SET
    target.author_id = source.previous_author_id,
    target.updated_at = current_timestamp()

### Cell 10: Queue restored works for reprocessing

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_phase1
WHERE work_first = author_first
  AND work_first IS NOT NULL AND work_first != ''
  AND LENGTH(work_first) > 2
  AND ( ARRAYS_OVERLAP(work_institutions, author_institution_ids)
     OR (work_middle IS NOT NULL AND author_middle IS NOT NULL
         AND LENGTH(TRANSLATE(work_middle, '.', '')) >= 1
         AND LENGTH(TRANSLATE(author_middle, '.', '')) >= 1
         AND SUBSTRING(TRANSLATE(work_middle, '.', ''), 1, 1)
           = SUBSTRING(TRANSLATE(author_middle, '.', ''), 1, 1)) )

### Cell 11: Post-rollback verification

In [ ]:
SELECT
  COUNT(*) AS restored_records,
  COUNT(DISTINCT c.author_id) AS restored_authors
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_candidates_phase1 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE c.work_first = c.author_first
  AND c.work_first IS NOT NULL AND c.work_first != ''
  AND LENGTH(c.work_first) > 2
  AND ( ARRAYS_OVERLAP(c.work_institutions, c.author_institution_ids)
     OR (c.work_middle IS NOT NULL AND c.author_middle IS NOT NULL
         AND LENGTH(TRANSLATE(c.work_middle, '.', '')) >= 1
         AND LENGTH(TRANSLATE(c.author_middle, '.', '')) >= 1
         AND SUBSTRING(TRANSLATE(c.work_middle, '.', ''), 1, 1)
           = SUBSTRING(TRANSLATE(c.author_middle, '.', ''), 1, 1)) )
  AND wa.author_id = c.author_id